In [178]:
import importlib
utils = importlib.import_module(".utils", "abt-question-ocr.codes")
import pandas as pd
import re

In [179]:
dir_data = utils.Config.DIR_RESULTS / "amazontextract"
questions_combined = pd.DataFrame()
answers_combined = pd.DataFrame()
for f in dir_data.glob('*.csv'):
    print(f.name)
    if f.name.endswith('_answers.csv'):
        answers = pd.read_csv(f, index_col=0)
        answers['Answer Source'] = f.name.replace('_answers.csv', '.pdf')
        answers_combined = pd.concat([answers_combined, answers])
    elif f.name.endswith('_answers_table.csv'):
        answers = pd.read_csv(f, index_col=0)
        answers['Answers'] = answers[['Guess', 'Actual']].apply(lambda x: x['Actual'] if not pd.isna(x['Actual']) else x['Guess'], axis=1)
        answers['Answer Source'] = f.name.replace('_answers_table.csv', '.pdf')
        answers = answers.drop(columns=['Guess', 'Actual'])
        answers = answers[~pd.isna(answers['Answers'])]
        answers_combined = pd.concat([answers_combined, answers])
    else:
        questions = pd.read_csv(f, index_col=0)
        questions['Question Source'] = f.name.replace('.csv', '.pdf')
        questions_combined = pd.concat([questions_combined, questions])

ABT #17_answers.csv
ABT 2006 recert #23.csv
24-2007-recert no answers.csv
ABT #17.csv
ABT RE No 22 2005.csv
recert 2002- MH group answers_answers_table.csv
ABT2002#19.csv
Document3.csv
ABT #18_answers.csv
Document2.csv
Document6.csv
Document7.csv
ABTexamMay52004 #21.csv
Document5.csv
Document4.csv
Document8.csv
recert18- 2001-MH answers_answers_table.csv
ABT2003#20.csv
ABT2001#18.csv
ABT #19.csv
ABT #18.csv
Document2_answers_table.csv
Document.csv
Document_answers_table.csv
Document7_answers_table.csv
ABT RE No 22 2005 missing page.csv
ABT2000#17.csv
ABT #20.csv


In [180]:
answers_combined[answers_combined['Exam'] == 'Exam booklet page no. 2001'] = 'Exam booklet page no. 2001A'

In [181]:
def mapVal(x):
    grps = re.findall(r'(Recertification(\ Exam)?\ (\d{4})\ Part\ ([A-Z]).*)|(Exam\ booklet\ page\ no\.\ (\d{4}[A-Z]*))', x)
    if len(grps) == 0 or len(grps[0]) != 6: return x
    if grps[0][0] != '':
        return ''.join(grps[0][2:4])
    return grps[0][5]

In [182]:
questions_combined['Exam'] = questions_combined['Exam'].map(mapVal)
answers_combined['Exam'] = answers_combined['Exam'].map(mapVal)

In [183]:
answers_combined_nonpdfs = pd.read_csv(utils.Config.DIR_RESULTS / 'combined_answers_from_nonpdfs.csv', index_col=0)
answers_combined_nonpdfs['Answers'] = answers_combined_nonpdfs['Answers'].str.strip().replace(',? +', ', ', regex=True)
answers_combined_nonpdfs

,Exam,Questions,Answers,Answer Source
0,2005A,1,C,ANSWERS RECERT EXAM 2005.xlsx
1,2005A,2,C,ANSWERS RECERT EXAM 2005.xlsx
2,2005A,3,D,ANSWERS RECERT EXAM 2005.xlsx
3,2005A,4,B,ANSWERS RECERT EXAM 2005.xlsx
4,2005A,5,B,ANSWERS RECERT EXAM 2005.xlsx
...,...,...,...,...
416,2007C,36,E,ABT Recert 24-2007 MH group Answers.doc
417,2007C,37,"A, E",ABT Recert 24-2007 MH group Answers.doc
418,2007C,38,C,ABT Recert 24-2007 MH group Answers.doc
419,2007C,39,C,ABT Recert 24-2007 MH group Answers.doc


In [184]:
answers_combined = pd.concat([answers_combined, answers_combined_nonpdfs]).reset_index(drop=True)
answers_combined

,Questions,Exam,Answers,Answer Source
0,1,2000A,C,ABT #17.pdf
1,2,2000A,D,ABT #17.pdf
2,3,2000A,B,ABT #17.pdf
3,4,2000A,B,ABT #17.pdf
4,5,2000A,D,ABT #17.pdf
...,...,...,...,...
1519,36,2007C,E,ABT Recert 24-2007 MH group Answers.doc
1520,37,2007C,"A, E",ABT Recert 24-2007 MH group Answers.doc
1521,38,2007C,C,ABT Recert 24-2007 MH group Answers.doc
1522,39,2007C,C,ABT Recert 24-2007 MH group Answers.doc


In [185]:
def extractIndex(x):
    res = re.findall(r'([0-9]{1,2})\..*', x)
    if len(res): return res[0]
    return '?'

In [186]:
questions_combined['Question Index'] = questions_combined[['Exam', 'Questions']].apply(lambda x: f"{x['Exam']} Q{extractIndex(x['Questions'])}", axis=1)
answers_combined['Question Index'] = answers_combined['Exam'] + ' Q' + answers_combined['Questions'].astype(str)

In [187]:
questions_combined.query('`Question Index` == "2001A Q38"')

,Exam,Questions,Options,Question Source,Question Index
33,2001A,38. Which of the following occupational lung d...,"1) byssinosis, 2) farmer's lung, 3) bagassosis...",Document7.pdf,2001A Q38
37,2001A,38. Which of the following occupational lung d...,"1) byssinosis, 2) farmer's lung, 3) bagassosis...",Document8.pdf,2001A Q38
37,2001A,38. Which of the following occupational lung d...,"1) byssinosis, 2) farmer's lung, 3) bagassosis...",ABT2001#18.pdf,2001A Q38


In [188]:
answers_combined.query('`Question Index` == "2001A Q38"')

,Questions,Exam,Answers,Answer Source,Question Index
337,38,2001A,A,ABT #18.pdf,2001A Q38


In [219]:
def multipleSelectionCriteria(answer):
    match answer:
        case 'A': return '1,2,3'
        case 'B': return '1,3'
        case 'C': return '2,4'
        case 'D': return '4'
        case 'E': return '1,2,3,4'
        case _: return answer

def format(x):
    def formatOptions(options):
        d_options = {}
        for opt in options.split(', '):
            if re.match(r'^[A-Z\d]\.', opt): 
                label, *txt = opt.split('.')
                d_options[label.strip()] = '.'.join(txt).strip()
            elif re.match(r'^\(?[A-Z]\)', opt):
                label, *txt = opt.split(')')
                d_options[label.strip().replace('(', '')] = ')'.join(txt).strip()
        return d_options
    
    question = '. '.join(x.iloc[0]['Questions'].split('. ')[1:]).strip()

    if pd.isna(x.iloc[0]['Options']): 
        options = {}
    else:
        try:
            options = formatOptions(x.iloc[0]['Options'])
        except Exception as exp:
            print(x)
            raise exp

    def formatAnswer(ans, options):
        return multipleSelectionCriteria(ans) if len(options) == 0 or re.match(r'^\d', str(list(options.keys())[0])) else ans
        
    q_src = ', '.join(x['Question Source'].unique())
    answer = sorted(set(x['Answers'].apply(lambda ans: tuple(formatAnswer(ans, options).split(','))).values))
    a_src = ', '.join(x['Answer Source'].unique())
    return pd.Series({'Questions': question, 'Options': options, 'Question Source': q_src, 'Answers': answer, 'Answer Source': a_src})

abt_qa = pd.merge(questions_combined[['Question Index', 'Questions', 'Options', 'Question Source']], answers_combined[['Answers', 'Question Index', 'Answer Source']], on='Question Index', how='inner')
abt_qa = abt_qa.drop_duplicates()
abt_qa = abt_qa.groupby('Question Index')[['Questions', 'Options', 'Question Source', 'Answers', 'Answer Source']].apply(lambda x: format(x)).reset_index()

def sortQA(x):
    df = x.str.split(' Q', expand=True)
    return df.apply(lambda x: [x[0], int(x[1])], axis=1)

abt_qa = abt_qa.sort_values(by='Question Index', key=sortQA)
abt_qa.reset_index(drop=True).to_csv(utils.Config.DIR_RESULTS / "abt_qa.csv")
abt_qa

,Question Index,Questions,Options,Question Source,Answers,Answer Source
0,2000A Q1,Which of the following pairs of radionuclides ...,"{'A': '222 Rn and 137 Cs', 'B': '222 Rn and 90...",ABT2000#17.pdf,"[(C,)]","ABT #17.pdf, Document.pdf"
11,2000A Q2,A semi-conscious pesticide sprayer is brought ...,{'A': 'propanolol followed by an intravenous a...,ABT2000#17.pdf,"[(D,)]","ABT #17.pdf, Document.pdf"
22,2000A Q3,Botulinum toxin inhibits neuromuscular communi...,{'A': 'blocking sodium channels along the moto...,ABT2000#17.pdf,"[(B,)]","ABT #17.pdf, Document.pdf"
32,2000A Q4,Some elements can be methylated by metabolic p...,"{'A': 'trimethyl tin', 'B': 'methyl mercury', ...",ABT2000#17.pdf,"[(B,), (C,)]","ABT #17.pdf, Document.pdf"
43,2000A Q5,Contact urticaria is characteristic of,"{'A': 'mistletoe', 'B': 'crocus', 'C': 'pokewe...","ABT #17.pdf, Document.pdf, ABT2000#17.pdf","[(D,)]","ABT #17.pdf, Document.pdf"
...,...,...,...,...,...,...
778,2007C Q36,"The following factors, identified by Bradford-...","{'A': 'consistency of the association', 'B': '...",24-2007-recert no answers.pdf,"[(E,)]",ABT Recert 24-2007 MH group Answers.doc
779,2007C Q37,The probability that a chemical may produce ca...,{'A': 'whether comparable peak plasma concentr...,24-2007-recert no answers.pdf,"[(A, E)]",ABT Recert 24-2007 MH group Answers.doc
780,2007C Q38,Elevation of which of the following serum/plas...,"{'A': 'troponin I', 'B': 'alkaline phosphatase...",24-2007-recert no answers.pdf,"[(C,)]",ABT Recert 24-2007 MH group Answers.doc
781,2007C Q39,Short Term Exposure Limits (STELs) are,{'A': 'the absolute maximum concentrations to ...,24-2007-recert no answers.pdf,"[(C,)]",ABT Recert 24-2007 MH group Answers.doc


In [235]:
def isNoisy(row):
    if len(row['Answers']) > 1: return True
    try:
        list(map(lambda x: f'({x.strip()}) {row['Options'][x.strip()]}', row['Answers'][0]))
    except:
        return True
    return False

is_noisy = abt_qa.apply(lambda x: isNoisy(x), axis=1)
abt_qa_wo_noise = abt_qa[~is_noisy]
abt_qa_wo_noise = abt_qa_wo_noise.reset_index(drop=True)
abt_qa_wo_noise.to_csv(utils.Config.DIR_RESULTS / "abt_qa_wo_noise.csv")
abt_qa_wo_noise.to_json(utils.Config.DIR_RESULTS / "abt_qa_wo_noise.json")

abt_qa[is_noisy]


,Question Index,Questions,Options,Question Source,Answers,Answer Source
32,2000A Q4,Some elements can be methylated by metabolic p...,"{'A': 'trimethyl tin', 'B': 'methyl mercury', ...",ABT2000#17.pdf,"[(B,), (C,)]","ABT #17.pdf, Document.pdf"
1,2000A Q10,Ingestion of which of the following radioactiv...,"{'A': 'beta emitters', 'B': 'alpha emitters', ...","ABT #17.pdf, Document.pdf, ABT2000#17.pdf","[(B,), (B, D), (C,)]","ABT #17.pdf, Document.pdf"
26,2000A Q33,Characteristics that distinguish pyrethroid ty...,{'A': 'mammalian toxicity is manifested by par...,Document.pdf,"[(A,), (E,)]","ABT #17.pdf, Document.pdf"
27,2000A Q34,Trivalent chromium,{'A': 'is an essential trace nutrient and cofa...,Document.pdf,"[(E,)]","ABT #17.pdf, Document.pdf"
33,2000A Q40,Which of the following substances exerts its p...,"{'1': 'botulinum toxin', '2': 'nicotine', '3':...",Document.pdf,"[(1, 2, 3), (1, 2, 3, 4)]","ABT #17.pdf, Document.pdf"
...,...,...,...,...,...,...
671,2006C Q47,The primary target organ(s) of toxicity observ...,{},ABT 2006 recert #23.pdf,"[(1, 3)]",ABT 2006 recert #23 answers.xlsx
672,2006C Q48,Which of the following is/are true?,{},ABT 2006 recert #23.pdf,"[(1, 2, 3)]",ABT 2006 recert #23 answers.xlsx
673,2006C Q49,Which of the following is/are correct regardin...,{},ABT 2006 recert #23.pdf,"[(1, 2, 3)]",ABT 2006 recert #23 answers.xlsx
675,2006C Q50,Emetics are contraindicated in which of the fo...,{},ABT 2006 recert #23.pdf,"[(1, 2, 3, 4)]",ABT 2006 recert #23 answers.xlsx
